# 阶段 5：GPU 真实候选搜索

本 Notebook 只使用 2010–2018 训练上下文和冻结 Reward，在 CUDA 上启动新 run 或恢复同一 run。验证集与最终样本外不会进入 Reward。所有产物写入已忽略的 `runs/real_search/<run_id>/`。

本次新建完整文法分层策略 run：依次预测元数族、六类文法子类、具体 Feature/Operator，并仅对时序算子条件预测 Window。保持 `max_depth=6`、`max_nodes=15`、原始 Reward、TB Loss、batch 8、`initial_log_z=39.0`、两组学习率和独立双 `max_norm=5.0` 不变。首段运行100步，与旧元数分组策略 run 的前100步按800个有效样本公平比较；验证集和最终样本外不参与。`PLANNED_MAX_STEPS` 会进入配置指纹，创建后不能修改。

## 1. 环境和项目导入

In [ ]:
import json
import platform
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_dir, *working_dir.parents) if (path / 'factor_gfn').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('无法从当前目录向上找到 factor_gfn 项目根目录')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from factor_gfn.gfn import (
    RealSearchSettings,
    create_real_search_runner,
    require_cuda_device,
    resume_real_search_runner,
)


## 2. 手工运行参数

新 run 使用 `MODE='new'`。恢复时只修改 `MODE`、`RESUME_RUN_DIR` 和 `TARGET_STEP`；恢复配置、seed、设备及数据指纹均从原 run 读取并严格校验。

In [ ]:
MODE = 'resume'  # 速度优化不改变模型或配置指纹，从当前grammar检查点续跑
DEVICE = 'cuda:0'
SEED = 42
PLANNED_MAX_STEPS = 1000  # 新 run 的工程上限；首次创建后不能修改
DIAGNOSTIC_STEPS = 10  # 仅验证批量诊断与新增分项计时的真实GPU收益
TARGET_STEP = None  # 创建或恢复后自动设置为 current_step + DIAGNOSTIC_STEPS
CHECKPOINT_INTERVAL = 10
CACHE_MAX_ENTRIES = 50_000
SUBEXPRESSION_CACHE_MAX_BYTES = 0  # 实测命中 3/6425，正式搜索默认关闭；实现仍保留供定向实验
RESUME_RUN_DIR = (
    PROJECT_ROOT / 'runs' / 'real_search'
    / 'd521789d86de425794a9e871b42db586'
)
BASELINE_RUN_DIR = (
    PROJECT_ROOT / 'runs' / 'real_search'
    / '3bb6e7af03b648a1a1f4ba51123554df'
)  # 历史元数分组策略100步 run，只读对照，不会恢复或改写

if MODE not in {'new', 'resume'}:
    raise ValueError("MODE 只能是 'new' 或 'resume'")
if MODE == 'new' and RESUME_RUN_DIR is not None:
    raise ValueError('新 run 不应设置 RESUME_RUN_DIR')
if MODE == 'resume' and RESUME_RUN_DIR is None:
    raise ValueError('恢复模式必须设置 RESUME_RUN_DIR')

resolved_device, cuda_environment = require_cuda_device(DEVICE)
display(pd.DataFrame([{
    'project_root': str(PROJECT_ROOT),
    'python': platform.python_version(),
    'numpy': np.__version__,
    'torch': torch.__version__,
    **cuda_environment,
}]))


## 3. 创建新 run 或严格恢复

In [ ]:
if MODE == 'new':
    settings = RealSearchSettings(
        max_steps=PLANNED_MAX_STEPS,
        seed=SEED,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        device=DEVICE,
        cache_max_entries=CACHE_MAX_ENTRIES,
        subexpression_cache_max_bytes=SUBEXPRESSION_CACHE_MAX_BYTES,
        run_root=PROJECT_ROOT / 'runs' / 'real_search',
    )
    runner = create_real_search_runner(settings)
else:
    runner = resume_real_search_runner(Path(RESUME_RUN_DIR))

config = runner.config
if TARGET_STEP is None:
    TARGET_STEP = min(
        runner.trainer.step + DIAGNOSTIC_STEPS,
        config.training.max_steps,
    )
assert config.search_space.max_depth == 6
assert config.search_space.max_nodes == 15
assert config.model.d_model == 128
assert config.model.num_heads == 4
assert config.model.num_layers == 4
assert config.model.dim_feedforward == 512
assert config.model.dropout == 0.0
assert config.model.token_policy_mode == 'grammar_hierarchical'
assert config.training.batch_size == 8
assert config.training.initial_log_z == 39.0
assert config.training.log_z_learning_rate == 1e-2
assert config.training.model_gradient_clip_norm == 5.0
assert config.training.log_z_gradient_clip_norm == 5.0
assert config.reward.candidate_industry_neutralization is True
assert runner.trainer.device.type == 'cuda'
assert runner.trainer.reward_provider.manifest()['calendar']['last_date'] <= '2018-12-31'
subexpression_cache = runner.trainer.reward_provider.manifest()['cache']['subexpression']
assert subexpression_cache['max_bytes'] == SUBEXPRESSION_CACHE_MAX_BYTES
assert subexpression_cache['enabled'] is False
assert subexpression_cache['eviction'] == 'lru'
assert runner.trainer.step < TARGET_STEP <= config.training.max_steps

display(pd.json_normalize(config.manifest()['config']))
display(pd.DataFrame([{
    'mode': MODE,
    'run_id': runner.trainer.run_id,
    'run_dir': str(runner.run_dir),
    'current_step': runner.trainer.step,
    'target_step': TARGET_STEP,
    'max_steps': config.training.max_steps,
    'config_fingerprint': config.fingerprint(),
    'provider_fingerprint': runner.recording_provider.fingerprint(),
    'subexpression_cache_mib': subexpression_cache['max_bytes'] / 1024**2,
}]))


## 4. 手动启动至目标步

该单元会执行真实训练。每一步都会原子更新完整候选、统计和 latest 检查点；每逢 `CHECKPOINT_INTERVAL` 或本次 `TARGET_STEP` 另存归档检查点。

In [ ]:
new_stats = runner.run_until(TARGET_STEP)
display(pd.DataFrame([asdict(item) for item in new_stats]))
print('运行目录：', runner.run_dir)
print('当前 step：', runner.trainer.step)
print('optimizer step：', runner.trainer.optimizer_step)


## 5. 运行结果与启动验收

In [ ]:
stats_payload = json.loads((runner.run_dir / 'training_stats.json').read_text(encoding='utf-8'))
history = pd.DataFrame(stats_payload['history'])
performance = pd.DataFrame(stats_payload['performance'])
display(history.tail(20))
display(performance.tail(20))

diagnostic_columns = [
    'step', 'loss', 'tb_delta_mean', 'tb_delta_std',
    'tb_delta_mean_square_ratio', 'tb_delta_std_square_ratio',
    'mean_log_pf', 'mean_log_pb', 'log_reward_mean',
    'gradient_norm', 'model_gradient_norm_before_clip',
    'log_z_gradient_before_clip',
    'model_gradient_clip_coefficient', 'log_z_gradient_clip_coefficient',
    'model_parameter_update_norm', 'model_relative_update_norm',
    'group_entropy_mean', 'group_entropy_normalized_mean',
    'leaf_group_probability_mean', 'unary_group_probability_mean',
    'binary_group_probability_mean', 'leaf_action_rate',
    'unary_action_rate', 'binary_action_rate',
    'grammar_category_entropy_mean', 'grammar_category_entropy_normalized_mean',
    'operator_entropy_mean', 'operator_entropy_normalized_mean',
    'window_entropy_mean', 'window_entropy_normalized_mean',
    'feature_category_action_rate', 'unary_category_action_rate',
    'ts_unary_category_action_rate', 'binary_category_action_rate',
    'ts_binary_category_action_rate', 'cross_sectional_category_action_rate',
    'window_5_action_rate', 'window_10_action_rate', 'window_20_action_rate',
    'window_40_action_rate', 'window_60_action_rate',
    'temporal_operator_action_rate',
    'terminal_node_count_p50', 'terminal_node_count_p90',
    'max_node_terminal_rate',
    'log_z', 'log_z_update',
]
diagnostic_history = (
    history.dropna(subset=['tb_delta_mean'])
    .tail(DIAGNOSTIC_STEPS)
    .copy()
)
if diagnostic_history.empty:
    raise RuntimeError('没有找到新增诊断字段，请确认已使用修改后的代码续跑')
display(diagnostic_history[diagnostic_columns])
diagnostic_summary = diagnostic_history[diagnostic_columns[1:]].agg(
    ['mean', 'min', 'max']
).T
diagnostic_summary['first_to_last'] = (
    diagnostic_history[diagnostic_columns[1:]].iloc[-1]
    - diagnostic_history[diagnostic_columns[1:]].iloc[0]
)
display(diagnostic_summary)

mean_ratio = float(diagnostic_history['tb_delta_mean_square_ratio'].mean())
std_ratio = float(diagnostic_history['tb_delta_std_square_ratio'].mean())
median_model_clip = float(diagnostic_history['model_gradient_clip_coefficient'].median())
median_log_z_clip = float(diagnostic_history['log_z_gradient_clip_coefficient'].median())
median_model_update = float(diagnostic_history['model_relative_update_norm'].median())
median_log_z_update = float(diagnostic_history['log_z_update'].abs().median())
median_abs_delta_mean = float(diagnostic_history['tb_delta_mean'].abs().median())
estimated_log_z_steps = (
    median_abs_delta_mean / median_log_z_update
    if median_log_z_update > 0.0
    else np.inf
)
diagnosis = {
    'diagnostic_steps_recorded': len(diagnostic_history),
    'tb_mean_offset_dominant': mean_ratio > 0.5,
    'tb_residual_variance_dominant': std_ratio > 0.5,
    'strong_model_gradient_clipping': median_model_clip < 0.01,
    'strong_log_z_gradient_clipping': median_log_z_clip < 0.01,
    'model_relative_update_very_small': (
        median_model_update < config.training.learning_rate * 0.1
    ),
    'estimated_log_z_steps_at_current_rate': estimated_log_z_steps,
    'log_z_calibration_too_slow_for_remaining_budget': (
        mean_ratio > 0.5
        and estimated_log_z_steps > config.training.max_steps - runner.trainer.step
    ),
}
display(pd.DataFrame([diagnosis]))

non_skipped = history[~history['skipped_update']] if len(history) else history
high_rejection = history[history['batch_rejection_rate'] > 0.80] if len(history) else history
acceptance = {
    'cuda_device': runner.trainer.device.type == 'cuda',
    'target_reached': runner.trainer.step == TARGET_STEP,
    'has_optimizer_update': runner.trainer.optimizer_step > 0,
    'finite_non_skipped_loss': bool(len(non_skipped) and np.isfinite(non_skipped['loss'].astype(float)).all()),
    'illegal_action_rate_zero': bool(len(history) and (history['illegal_action_rate'] == 0.0).all()),
    'latest_checkpoint_exists': runner.latest_checkpoint_path.is_file(),
    'evaluations_exist': runner.evaluations_path.is_file(),
    'oos_not_exposed': runner.trainer.reward_provider.manifest()['calendar']['last_date'] <= '2018-12-31',
}
display(pd.DataFrame([acceptance]))
assert all(acceptance.values()), acceptance
if len(high_rejection):
    print('警告：存在拒绝率超过 80% 的步骤。请先分析 evaluations.jsonl，不自动修改 Reward 或训练配置。')

def _load_main_candidate_frame(run_dir):
    rows = []
    with (Path(run_dir) / 'evaluations.jsonl').open(encoding='utf-8') as handle:
        for line in handle:
            record = json.loads(line)
            if record.get('branch') != 'main':
                continue
            result = record.get('metadata', {}).get('reward_result') or {}
            rows.append({
                'logical_step': int(record['logical_step']),
                'structural_hash': record['structural_hash'],
                'node_count': int(record['node_count']),
                'valid': bool(record.get('valid')),
                'reward': record.get('reward'),
                'train_ic': result.get('train_ic'),
                'train_long_ir': result.get('train_long_ir'),
                'barra_ts_corr': result.get('barra_ts_corr'),
            })
    return pd.DataFrame(rows)

def _candidate_quality_row(frame, label, first_step=1, last_step=100):
    window = frame[frame['logical_step'].between(first_step, last_step)].copy()
    valid = window[window['valid']].copy()
    valid['reward'] = pd.to_numeric(valid['reward'], errors='coerce')
    valid['abs_train_ic'] = pd.to_numeric(valid['train_ic'], errors='coerce').abs()
    valid['train_long_ir'] = pd.to_numeric(valid['train_long_ir'], errors='coerce')
    valid['barra_ts_corr'] = pd.to_numeric(valid['barra_ts_corr'], errors='coerce')
    unique_valid = valid.sort_values('reward', ascending=False).drop_duplicates('structural_hash')
    train_side = unique_valid[
        (unique_valid['abs_train_ic'] > 0.01)
        & (unique_valid['train_long_ir'] > 0.25)
        & (unique_valid['barra_ts_corr'] < 0.7)
    ]
    high_ic = unique_valid[unique_valid['abs_train_ic'] > 0.03]
    denominator = max(len(unique_valid), 1)
    return {
        'run': label,
        'steps': last_step - first_step + 1,
        'requests': len(window),
        'valid_requests': len(valid),
        'unique_valid_structures': len(unique_valid),
        'unique_valid_rate': len(unique_valid) / max(len(valid), 1),
        'rejection_rate': 1.0 - len(valid) / max(len(window), 1),
        'reward_p90': float(unique_valid['reward'].quantile(0.90)),
        'reward_p99': float(unique_valid['reward'].quantile(0.99)),
        'reward_max': float(unique_valid['reward'].max()),
        'abs_train_ic_p90': float(unique_valid['abs_train_ic'].quantile(0.90)),
        'abs_train_ic_max': float(unique_valid['abs_train_ic'].max()),
        'abs_train_ic_gt_003': len(high_ic),
        'abs_train_ic_gt_003_per_1000': len(high_ic) / denominator * 1000.0,
        'train_side_screen': len(train_side),
        'train_side_screen_per_1000': len(train_side) / denominator * 1000.0,
        'terminal_nodes_mean': float(unique_valid['node_count'].mean()),
        'terminal_nodes_p50': float(unique_valid['node_count'].quantile(0.50)),
        'terminal_nodes_p90': float(unique_valid['node_count'].quantile(0.90)),
        'max_node_terminal_rate': float((unique_valid['node_count'] == 15).mean()),
    }

def _training_dynamics_row(label, run_dir):
    payload = json.loads((Path(run_dir) / 'training_stats.json').read_text(encoding='utf-8'))
    frame = pd.DataFrame(payload['history']).head(100)
    return {
        'run': label,
        'steps': len(frame),
        'effective_samples': int(frame['effective_batch_size'].sum()),
        'loss_mean': float(frame['loss'].mean()),
        'delta_mean': float(frame['tb_delta_mean'].mean()),
        'delta_std': float(frame['tb_delta_std'].mean()),
        'entropy_normalized': float(frame['policy_entropy_normalized_mean'].mean()),
        'rejection_rate': float(frame['batch_rejection_rate'].mean()),
        'group_entropy_normalized': (
            float(frame['group_entropy_normalized_mean'].mean())
            if 'group_entropy_normalized_mean' in frame else np.nan
        ),
        'leaf_group_probability': (
            float(frame['leaf_group_probability_mean'].mean())
            if 'leaf_group_probability_mean' in frame else np.nan
        ),
        'unary_group_probability': (
            float(frame['unary_group_probability_mean'].mean())
            if 'unary_group_probability_mean' in frame else np.nan
        ),
        'binary_group_probability': (
            float(frame['binary_group_probability_mean'].mean())
            if 'binary_group_probability_mean' in frame else np.nan
        ),
        'leaf_action_rate': (
            float(frame['leaf_action_rate'].mean())
            if 'leaf_action_rate' in frame else np.nan
        ),
        'unary_action_rate': (
            float(frame['unary_action_rate'].mean())
            if 'unary_action_rate' in frame else np.nan
        ),
        'binary_action_rate': (
            float(frame['binary_action_rate'].mean())
            if 'binary_action_rate' in frame else np.nan
        ),
    }

if not BASELINE_RUN_DIR.is_dir():
    raise FileNotFoundError(f'找不到旧元数分组100步 run：{BASELINE_RUN_DIR}')
arity_candidates = _load_main_candidate_frame(BASELINE_RUN_DIR)
grammar_candidates = _load_main_candidate_frame(runner.run_dir)
quality = pd.DataFrame([
    _candidate_quality_row(arity_candidates, 'arity_steps_001_100'),
    _candidate_quality_row(grammar_candidates, 'grammar_steps_001_100'),
]).set_index('run')
display(quality)
display((
    quality.loc['grammar_steps_001_100']
    - quality.loc['arity_steps_001_100']
).to_frame('grammar_minus_arity'))
dynamics = pd.DataFrame([
    _training_dynamics_row('arity_steps_001_100', BASELINE_RUN_DIR),
    _training_dynamics_row('grammar_steps_001_100', runner.run_dir),
]).set_index('run')
display(dynamics)
assert dynamics['effective_samples'].eq(800).all(), dynamics['effective_samples']
assert history['group_entropy_mean'].notna().all()
assert history['grammar_category_entropy_mean'].notna().all()
assert history['operator_entropy_mean'].notna().all()
assert history['window_entropy_mean'].notna().any()
assert np.allclose(
    history[['leaf_action_rate', 'unary_action_rate', 'binary_action_rate']].sum(axis=1),
    1.0,
)
assert np.allclose(
    history[[
        'feature_category_action_rate', 'unary_category_action_rate',
        'ts_unary_category_action_rate', 'binary_category_action_rate',
        'ts_binary_category_action_rate', 'cross_sectional_category_action_rate',
    ]].sum(axis=1),
    1.0,
)
window_rows = history['temporal_operator_action_rate'].fillna(0.0) > 0.0
assert window_rows.any()
assert np.allclose(
    history.loc[window_rows, [
        'window_5_action_rate', 'window_10_action_rate', 'window_20_action_rate',
        'window_40_action_rate', 'window_60_action_rate',
    ]].sum(axis=1),
    1.0,
)
print('完整文法分层策略100步同样本量对照完成。只使用训练期候选；验证集与OOS未参与。')
